Machine Learning Assignment 1 - Analytical Reasoning

In [40]:
# Import all required libraries

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from sklearn.decomposition import PCA

In [41]:
# Load the dataset

df = pd.read_excel("Clinical_Risk_v2.xlsx")
df.head()

,patient_id,age,blood_pressure,cholesterol_level,bmi,smoking_status,physical_activity,family_history,diet_score,registration_month,risk_status
0,1,25,115,261.0,18.2,Former,Medium,Yes,6.1,Jun,0
1,2,67,147,297.0,30.5,Never,High,Yes,3.4,May,1
2,3,65,150,160.0,19.1,Current,Low,No,7.0,Oct,1
3,4,60,129,171.0,36.0,Former,High,Yes,3.0,Jan,1
4,5,62,157,264.0,39.5,Never,High,No,6.8,Aug,1


In [42]:
# Inspect dataset structure and missing values

df.info()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   patient_id          500 non-null    int64  
 1   age                 500 non-null    int64  
 2   blood_pressure      500 non-null    int64  
 3   cholesterol_level   460 non-null    float64
 4   bmi                 474 non-null    float64
 5   smoking_status      500 non-null    object 
 6   physical_activity   500 non-null    object 
 7   family_history      500 non-null    object 
 8   diet_score          500 non-null    float64
 9   registration_month  500 non-null    object 
 10  risk_status         500 non-null    int64  
dtypes: float64(3), int64(4), object(4)
memory usage: 43.1+ KB


patient_id             0
age                    0
blood_pressure         0
cholesterol_level     40
bmi                   26
smoking_status         0
physical_activity      0
family_history         0
diet_score             0
registration_month     0
risk_status            0
dtype: int64

In [43]:
# Handle missing values for cholesterol_level and bmi

# Impute missing numeric values with the median of each column
df['cholesterol_level'] = df['cholesterol_level'].fillna(df['cholesterol_level'].median())
df['bmi'] = df['bmi'].fillna(df['bmi'].median())

# Verify no missing values remain
df.isnull().sum()

patient_id            0
age                   0
blood_pressure        0
cholesterol_level     0
bmi                   0
smoking_status        0
physical_activity     0
family_history        0
diet_score            0
registration_month    0
risk_status           0
dtype: int64

In [44]:
# Define features and target variable

X = df.drop(columns=['risk_status', 'patient_id'])
y = df['risk_status']

In [45]:
# Spliting the data into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [46]:
# Identify numeric and categorical feature columns

numeric_features = ['age', 'blood_pressure', 'cholesterol_level', 'bmi', 'diet_score']
categorical_features = ['smoking_status', 'physical_activity', 'family_history', 'registration_month']


In [47]:
# Build preprocessing transformer (scaling + one‑hot encoding)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)


In [48]:
# Build Logistic Regression model pipeline

log_reg_model = Pipeline(
    steps=[
        ('preprocess', preprocessor),
        ('model', LogisticRegression(max_iter=1000))
    ]
)


In [49]:
# Training the Logistic Regression model

log_reg_model.fit(X_train, y_train)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers cont

In [50]:
# Evaluate Logistic Regression model

y_pred_log = log_reg_model.predict(X_test)

log_accuracy = accuracy_score(y_test, y_pred_log)
log_precision = precision_score(y_test, y_pred_log)
log_recall = recall_score(y_test, y_pred_log)
log_f1 = f1_score(y_test, y_pred_log)

print(f"log_accuracy: {log_accuracy}")
print(f"log_precision: {log_precision}")
print(f"log_recall: {log_recall}")
print(f"log_f1: {log_f1}")

log_accuracy: 0.89
log_precision: 0.9125
log_recall: 0.948051948051948
log_f1: 0.9299363057324841


In [51]:
# Build Decision Tree model pipeline

dt_model = Pipeline(
    steps=[
        ('preprocess', preprocessor),
        ('model', DecisionTreeClassifier(random_state=42))
    ]
)


In [52]:
# Training the Decision Tree model

dt_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers cont

In [53]:
# Evaluate Decision Tree model

y_pred_dt = dt_model.predict(X_test)

dt_accuracy = accuracy_score(y_test, y_pred_dt)
dt_precision = precision_score(y_test, y_pred_dt)
dt_recall = recall_score(y_test, y_pred_dt)
dt_f1 = f1_score(y_test, y_pred_dt)

print(f"dt_accuracy: {dt_accuracy}")
print(f"dt_precision: {dt_precision}")
print(f"dt_recall: {dt_recall}")
print(f"dt_f1: {dt_f1}")

dt_accuracy: 0.87
dt_precision: 0.9210526315789473
dt_recall: 0.9090909090909091
dt_f1: 0.9150326797385621


In [54]:
# Apply PCA (2 components) on the preprocessed training data

# First fit the preprocessor on training data
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Apply PCA
pca = PCA(n_components=2)
X_train_pca = pca.fit_transform(X_train_processed)
X_test_pca = pca.transform(X_test_processed)


In [55]:
# Train Logistic Regression on PCA-transformed data
log_reg_pca = LogisticRegression(max_iter=1000)
log_reg_pca.fit(X_train_pca, y_train)

y_pred_log_pca = log_reg_pca.predict(X_test_pca)

log_pca_accuracy = accuracy_score(y_test, y_pred_log_pca)
log_pca_precision = precision_score(y_test, y_pred_log_pca)
log_pca_recall = recall_score(y_test, y_pred_log_pca)
log_pca_f1 = f1_score(y_test, y_pred_log_pca)

print(f"log_pca_accuracy: {log_pca_accuracy}")
print(f"log_pca_precision: {log_pca_precision}")
print(f"log_pca_recall: {log_pca_recall}")
print(f"log_pca_f1: {log_pca_f1}")

log_pca_accuracy: 0.77
log_pca_precision: 0.77
log_pca_recall: 1.0
log_pca_f1: 0.8700564971751412


In [56]:
# Train Decision Tree on PCA-transformed data
dt_pca = DecisionTreeClassifier(random_state=42)
dt_pca.fit(X_train_pca, y_train)

y_pred_dt_pca = dt_pca.predict(X_test_pca)

dt_pca_accuracy = accuracy_score(y_test, y_pred_dt_pca)
dt_pca_precision = precision_score(y_test, y_pred_dt_pca)
dt_pca_recall = recall_score(y_test, y_pred_dt_pca)
dt_pca_f1 = f1_score(y_test, y_pred_dt_pca)

print(f"dt_pca_accuracy: {dt_pca_accuracy}")
print(f"dt_pca_precision: {dt_pca_precision}")
print(f"dt_pca_recall: {dt_pca_recall}")
print(f"dt_pca_f1: {dt_pca_f1}")

dt_pca_accuracy: 0.62
dt_pca_precision: 0.7671232876712328
dt_pca_recall: 0.7272727272727273
dt_pca_f1: 0.7466666666666667
